This notebook prepares the bottle-only subset from COCO and writes bottle.yaml.

In [ ]:
pip install -U onnx-graphsurgeon==0.5.2 sng4onnx==1.0.0 onnx2tf ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.8/198.8 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of onnx2tf to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.0/195.0 kB 20.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.2/199.2 kB 20.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.3/196.3 kB 20.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.3/196.3 kB 18.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 20.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.6/191.6 kB 21.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.9/191.9 kB 19.5 MB/s eta 0:00:00
INFO: pip is still looking at multiple versions of onnx2tf to determine which version is compatible with other requirements. Thi

In [1]:
!pip install -q pycocotools tqdm

In [ ]:
from pathlib import Path

# Adjust these if your paths differ
COCO_ROOT = Path("/content")
IMG_TRAIN = COCO_ROOT / "train2017"
IMG_VAL   = COCO_ROOT / "val2017"
ANN_TRAIN = COCO_ROOT / "annotations/instances_train2017.json"
ANN_VAL   = COCO_ROOT / "annotations/instances_val2017.json"

OUT_ROOT = Path("/content/bottle_dataset")

In [ ]:
from pycocotools.coco import COCO
from tqdm import tqdm
import shutil

def extract_bottle_split(
    coco_json,
    img_dir,
    out_img_dir,
    out_lbl_dir,
    max_images=None
):
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_lbl_dir.mkdir(parents=True, exist_ok=True)

    coco = COCO(coco_json)

    # Get bottle category ID
    cat_ids = coco.getCatIds(catNms=["bottle"])
    assert len(cat_ids) == 1, "Bottle category not found"
    bottle_id = cat_ids[0]

    # Get all image IDs that contain bottles
    img_ids = coco.getImgIds(catIds=[bottle_id])
    if max_images:
        img_ids = img_ids[:max_images]

    for img_id in tqdm(img_ids, desc=f"Extracting {out_img_dir.name}"):
        img_info = coco.loadImgs(img_id)[0]
        file_name = img_info["file_name"]
        w, h = img_info["width"], img_info["height"]

        # Copy image
        src_img = img_dir / file_name
        dst_img = out_img_dir / file_name
        if not dst_img.exists():
            shutil.copy(src_img, dst_img)

        # Create YOLO label file
        ann_ids = coco.getAnnIds(imgIds=[img_id], catIds=[bottle_id], iscrowd=False)
        anns = coco.loadAnns(ann_ids)

        label_lines = []
        for ann in anns:
            x, y, bw, bh = ann["bbox"]

            # Convert COCO bbox → YOLO (normalized)
            cx = (x + bw / 2) / w
            cy = (y + bh / 2) / h
            bw /= w
            bh /= h

            label_lines.append(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        lbl_path = out_lbl_dir / (Path(file_name).stem + ".txt")
        lbl_path.write_text("\n".join(label_lines))

In [ ]:
# Train split
extract_bottle_split(
    coco_json=ANN_TRAIN,
    img_dir=IMG_TRAIN,
    out_img_dir=OUT_ROOT / "images/train",
    out_lbl_dir=OUT_ROOT / "labels/train",
    max_images=500
)

# Val split
extract_bottle_split(
    coco_json=ANN_VAL,
    img_dir=IMG_VAL,
    out_img_dir=OUT_ROOT / "images/val",
    out_lbl_dir=OUT_ROOT / "labels/val",
    max_images=100
)


loading annotations into memory...
Done (t=22.98s)
creating index...
index created!


Extracting train: 100%|██████████| 500/500 [00:02<00:00, 221.58it/s]


loading annotations into memory...
Done (t=0.98s)
creating index...
index created!


Extracting val: 100%|██████████| 100/100 [00:00<00:00, 334.27it/s]


In [ ]:
yaml_text = f"""
path: {OUT_ROOT}
train: images/train
val: images/val

names:
  0: bottle
""".strip()

(OUT_ROOT / "bottle.yaml").write_text(yaml_text)
print("Wrote:", OUT_ROOT / "bottle.yaml")

Wrote: /content/bottle_dataset/bottle.yaml


In [ ]:
print("Train images:", len(list((OUT_ROOT / "images/train").glob("*.jpg"))))
print("Train labels:", len(list((OUT_ROOT / "labels/train").glob("*.txt"))))
print("Val images:", len(list((OUT_ROOT / "images/val").glob("*.jpg"))))
print("Val labels:", len(list((OUT_ROOT / "labels/val").glob("*.txt"))))

Train images: 500
Train labels: 500
Val images: 100
Val labels: 100


In [ ]:
!cd /content
!zip -r bottle_dataset.zip bottle_dataset


zip error: Nothing to do! (bottle_dataset.zip)
